# Random Search Experiment

This notebook runs a random search over the hyperparameter space for the triangle-based evolutionary algorithm.

Rather than evaluating every combination, it samples N_SAMPLES random configurations from `SEARCH_SPACE` and runs N_TRIALS deterministic trials per configuration.

The project minimizes fitness: lower RMSE is better. Fitness sharing is fixed on for every run as the baseline diversity-preservation mechanism. Restricted mating is part of the search space, so the experiment tests whether different mating restrictions add value when fitness sharing is already active.

The expensive execution cell is guarded by `RUN_EXPERIMENT = False`. When enabled, each trial uses early stopping with patience 5 and saves one JSON file per trial to `results/random_search/` so interrupted runs resume safely.

In [1]:
%config InlineBackend.figure_format = 'retina'

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src import load_image
from src.ga import cross_over, fitness, grid_search, mutate

## Project API Connection

Reusable grid-search mechanics live in `src/ga/grid_search.py`. This notebook keeps only the experiment configuration, execution guard, and analysis cells.


In [2]:
FITNESS_FUNCTION = fitness.compute_rmse
MUTATION_FUNCTION = mutate.random_triangle_mutation
CROSSOVER_FUNCTIONS = {
    "two_point_one_child": cross_over.two_point_crossover,
    "two_point_two_children": cross_over.two_point_crossover_two_children,
    "pmx": cross_over.pmx_crossover,
}

## Fixed Parameters

These values stay constant for every run. The alpha range uses the current fixed-alpha setting already used in the project notebooks.


In [3]:
BASE_SEED = 73
N_TRIALS = 5
N_SAMPLES = 12
FITNESS_OBJECTIVE = "minimize"
EARLY_STOPPING_PATIENCE = 50
EARLY_STOPPING_MIN_DELTA = 0.01

RESULTS_DIR = project_root / "results"
SEARCH_RESULTS_DIR = RESULTS_DIR / "random_search"
SUMMARY_PATH = RESULTS_DIR / "random_search_summary.csv"
IMAGE_PATH = project_root / "images" / "girl_pearl_earing.png"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FIXED_PARAMS = {
    "elitism": 5,
    "population_size": 250,
    "generations": 500,
    "triangle_alpha_range": (255, 255),
    "fitness_sharing": True,
    "sigma_share": 0.3,
    "n_bins": 8,
    "candidate_pool": 5,
}

FIXED_PARAMS

{'elitism': 5,
 'population_size': 250,
 'generations': 500,
 'triangle_alpha_range': (255, 255),
 'fitness_sharing': True,
 'sigma_share': 0.3,
 'n_bins': 8,
 'candidate_pool': 5}

## Search Space

The random search evaluates every combination in this space.


In [4]:
SEARCH_SPACE = {
    "mutation_rate": [0.1],
    "crossover_rate": [0.9],
    "crossover_type": [
        "two_point_one_child",
        "two_point_two_children",
        "pmx",
    ],
    "selection_type": ["tournament", "ranking"],
    "restricted_mating": ["best_partial_match", "unidirectional"],
}

SEARCH_SPACE

{'mutation_rate': [0.1],
 'crossover_rate': [0.9],
 'crossover_type': ['two_point_one_child', 'two_point_two_children', 'pmx'],
 'selection_type': ['tournament', 'ranking'],
 'restricted_mating': ['best_partial_match', 'unidirectional']}

## Experiment Initialization

Build the complete random sample of experimental configurations, initialize reproducible setup ordering and trial seeds, load the target image, and prepare the result schema used throughout the experiment pipeline.

setup_id values are assigned sequentially according to the deterministic grid order, while trial seeds follow: `seed = 73 + setup_id * 100 + trial_id`


In [5]:
# --- Generate full random sample of experimental configurations and load target array --- #
random_setups = grid_search.build_random_setups(SEARCH_SPACE, n_samples=N_SAMPLES, seed=BASE_SEED)
EXPECTED_RUNS = len(random_setups) * N_TRIALS

# Load target array
target_array = load_image.load_target_image(IMAGE_PATH)

print(
    f"Target array: {target_array.shape} -> (H, W, 3) array with RGB values in [0, 255]"
)
print(f"Built {len(random_setups)} grid setups for {EXPECTED_RUNS} expected trial runs.")

Target array: (400, 300, 3) -> (H, W, 3) array with RGB values in [0, 255]
Built 12 grid setups for 60 expected trial runs.


## Run Experiment

Change `RUN_EXPERIMENT` to `True` when you are ready to launch the full run. The expected total run count is computed from the grid size.


In [ ]:
RUN_EXPERIMENT = True

if RUN_EXPERIMENT:
    for _, setup in random_setups.iterrows():
        setup_id = int(setup["setup_id"])
        for trial_id in range(1, N_TRIALS + 1):
            print(f"Setup {setup_id}/{len(random_setups)}, trial {trial_id}/{N_TRIALS}...")
            result = grid_search.run_one_trial(
                setup=setup,
                trial_id=trial_id,
                target_array=target_array,
                fixed_params=FIXED_PARAMS,
                results_dir=SEARCH_RESULTS_DIR,
                fitness_function=FITNESS_FUNCTION,
                mutation_function=MUTATION_FUNCTION,
                crossover_functions=CROSSOVER_FUNCTIONS,
                early_stopping_patience=EARLY_STOPPING_PATIENCE,
                early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
            )
            print(
                f"  fitness={result['final_best_fitness']:.6f}, "
                f"generations={result['generations_run']}, "
                f"stopped_early={result['stopped_early']}, "
                f"runtime={result['runtime_seconds']:.1f}s"
            )
else:
    print("RUN_EXPERIMENT is False. Set it to True to run or resume the experiment.")

Setup 1/12, trial 1/5...
  ↩ [search-pmx-tournament-unidirectional-mut0.1-xo0.9] trial 1 loaded from cache
  fitness=0.274214, generations=18, stopped_early=True, runtime=13.3s
Setup 1/12, trial 2/5...
  ↩ [search-pmx-tournament-unidirectional-mut0.1-xo0.9] trial 2 loaded from cache
  fitness=0.251995, generations=25, stopped_early=True, runtime=18.5s
Setup 1/12, trial 3/5...
  ↩ [search-pmx-tournament-unidirectional-mut0.1-xo0.9] trial 3 loaded from cache
  fitness=0.249365, generations=36, stopped_early=True, runtime=26.6s
Setup 1/12, trial 4/5...
  ↩ [search-pmx-tournament-unidirectional-mut0.1-xo0.9] trial 4 loaded from cache
  fitness=0.251117, generations=34, stopped_early=True, runtime=26.1s
Setup 1/12, trial 5/5...
  ↩ [search-pmx-tournament-unidirectional-mut0.1-xo0.9] trial 5 loaded from cache
  fitness=0.227391, generations=78, stopped_early=True, runtime=59.5s
Setup 2/12, trial 1/5...
  ↩ [search-two_point_one_child-tournament-best_partial_match-mut0.1-xo0.9] trial 1 loaded

## Aggregate Results

The summary is sorted for the project's objective direction. Because RMSE fitness is minimized, lower mean final fitness is better.


In [ ]:
raw_results = grid_search.load_all_results(SEARCH_RESULTS_DIR)

if raw_results.empty:
    summary = pd.DataFrame()
    print(
        "No random search results found yet. Set RUN_EXPERIMENT = True and run the experiment cell."
    )
else:
    summary = grid_search.build_summary(raw_results)
    summary.to_csv(SUMMARY_PATH, index=False)
    print(f"Saved summary to {SUMMARY_PATH}")
    display(summary.head(10))

## Plots

These plots are conditional: they render only after raw results exist.


In [ ]:
raw_results = grid_search.load_all_results(SEARCH_RESULTS_DIR)

if raw_results.empty:
    print(
        "No random search results found yet. Set RUN_EXPERIMENT = True and run the experiment cell."
    )
else:
    summary = grid_search.build_summary(raw_results)

    # Short label: crossover / selection / restricted_mating
    summary["setup_label"] = (
        summary["crossover_type"].str.replace("_", "\n")
        + "\n" + summary["selection_type"]
        + "\n" + summary["restricted_mating"].str.replace("_", "\n")
    )

    plot_summary    = summary.sort_values("mean_final_best_fitness", ascending=True).copy()
    runtime_summary = summary.sort_values("mean_runtime_seconds",    ascending=False).copy()

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle("Random Search Results", fontweight="bold")

    axes[0, 0].bar(
        plot_summary["setup_label"],
        plot_summary["mean_final_best_fitness"],
        color="steelblue",
        edgecolor="white",
    )
    axes[0, 0].set_title("Mean Fitness by Config (sorted best → worst)")
    axes[0, 0].set_xlabel("Config")
    axes[0, 0].set_ylabel("Mean final best fitness")
    axes[0, 0].tick_params(axis="x", rotation=45, labelsize=7)

    raw_results.boxplot(
        column="final_best_fitness",
        by="crossover_type",
        ax=axes[0, 1],
        grid=False,
        rot=20,
    )
    axes[0, 1].set_title("Fitness Distribution by Crossover Type")
    axes[0, 1].set_xlabel("Crossover type")
    axes[0, 1].set_ylabel("Final best fitness")

    raw_results.boxplot(
        column="final_best_fitness",
        by="selection_type",
        ax=axes[1, 0],
        grid=False,
    )
    axes[1, 0].set_title("Fitness Distribution by Selection Type")
    axes[1, 0].set_xlabel("Selection type")
    axes[1, 0].set_ylabel("Final best fitness")

    axes[1, 1].bar(
        runtime_summary["setup_label"],
        runtime_summary["mean_runtime_seconds"],
        color="darkorange",
        edgecolor="white",
    )
    axes[1, 1].set_title("Runtime by Config (sorted slowest → fastest)")
    axes[1, 1].set_xlabel("Config")
    axes[1, 1].set_ylabel("Mean runtime seconds")
    axes[1, 1].tick_params(axis="x", rotation=45, labelsize=7)

    plt.suptitle("Random Search Results", fontweight="bold")
    plt.tight_layout()
    plt.show()